In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/Preprocessing1.csv")
df.sample(5)
df.shape

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Month,WeekDay,Day
7314,Air Asia,Delhi,Cochin,DEL → BLR → COK,20:45,07:10 16 Jun,10h 25m,1 stop,No info,6253,6,Saturday,15.0
2741,Vistara,Chennai,Banglore,MAA → CCU,17:30,20:05,26h 55m,non-stop,No info,11982,5,Wednesday,15.0
2827,Jet Airways,Kolkata,Banglore,CCU → BOM → BLR,20:00,16:20 13 Jun,20h 20m,1 stop,In-flight meal not included,8529,6,Wednesday,12.0
1527,Air India,Mumbai,Cochin,BOM → HYD,17:50,22:25,7h 40m,non-stop,No info,3100,5,Saturday,NaN
3296,Jet Airways,Delhi,Cochin,DEL → BOM → COK,17:30,04:25 22 Mar,10h 55m,1 stop,In-flight meal not included,5963,3,Thursday,21.0


(9450, 13)

### MetaData
- Airline:The name of the airline 
- Source: The source from which the service begins 
- Destination: The destination where the service ends 
- Route: Route the flight took. 
- Dep_Time: The time when the journey starts from the source. 
- Arrival_Time: Time of arrival at the destination. 
- Duration: Total duration of the flight. 
- Total_Stops: Total stops between the source and destination. 
- Additional_Info: Additional information about the flight 
- Price: The price of the ticket ▶▶▶ Target 
- Month: Month of journey. 
- WeekDay: Day at which journey started. 
- Day: Date of the start of journey.

In [3]:
df['Price'].mean()

np.float64(9027.895555555555)

In [4]:
months = df['Month'].value_counts()
months
months.index[months.argmax()]

Month
5    3092
6    3044
3    2388
4     926
Name: count, dtype: int64

np.int64(5)

In [5]:
mask = df['WeekDay'].isin(['Saturday', 'Sunday'])
df[mask]["Price"].mean()
df[~mask]["Price"].mean()

np.float64(9058.016077170418)

np.float64(9015.219666215608)

In [6]:
mask = df['Additional_Info'] == 'No Info'
df.loc[mask, 'Additional_Info'] = 'No info'
((df['Airline'] == 'IndiGo') & (df['Additional_Info'] == 'No info')).sum()

np.int64(1650)

In [7]:
def convert_to_seconds(duration):
  def in_seconds(part):
    map = {
        "h": 3600, "m": 60, "s": 1
    }
    num, unit = int(part[:-1]), part[-1]
    return num * map[unit]

  return sum(in_seconds(part) for part in duration.split())


print(convert_to_seconds('10h 20m'))
print(convert_to_seconds('10h 20m 10s'))
print(convert_to_seconds('10h'))
print(convert_to_seconds('10m'))
print(convert_to_seconds('10s'))
print(convert_to_seconds('5m 10s'))
print(convert_to_seconds('1h 10s'))

37200
37210
36000
600
10
310
3610


In [8]:
df['Duration'].map(convert_to_seconds).mean()

np.float64(38957.93650793651)

In [9]:
def time_to_category(time):
  hour = int(time[:2])
  if 5 <= hour < 12:
    return 'Morning'
  elif 12 <= hour < 17:
    return 'Afternoon'
  elif 17 <= hour < 20:
    return 'Evening'
  else:
    return 'Night'


time_to_category("04:25 10 Jun")

'Night'

In [10]:
temp = df[['Dep_Time', 'Arrival_Time']].map(time_to_category)

((temp['Dep_Time'] == 'Morning') &
    (temp['Arrival_Time'] == 'Evening')).sum()

np.int64(922)